# Wolf Sheep Simple 3 — Narrativa e Simulação em Python

Este notebook reproduz, em Python, a lógica do modelo **Wolf Sheep Simple 3**, inspirado nos modelos de agentes do NetLogo.

A simulação representa um ecossistema em uma grade bidimensional, onde:

- **ovelhas** se movem, gastam energia e comem grama;
- **lobos** se movem, gastam energia e caçam ovelhas;
- **grama** cresce gradualmente nos patches;
- agentes morrem quando ficam sem energia;
- agentes podem se reproduzir conforme uma probabilidade definida;
- a dinâmica global emerge das interações locais entre agentes e ambiente.

O objetivo é observar como **predação**, **recurso limitado**, **energia**, **movimento** e **reprodução** produzem oscilações populacionais ou extinções.

## 1. Modelo original em NetLogo

O trecho NetLogo informado define uma raça de agentes chamada `sheep`, uma variável de energia para as ovelhas e uma variável de grama nos patches.

Nesta versão em Python, o modelo foi ampliado para o padrão **Wolf Sheep Simple 3**, incluindo também os lobos.

A estrutura conceitual é:

1. Cada agente ocupa uma posição `(x, y)` na grade.
2. Ovelhas comem grama e ganham energia.
3. Lobos comem ovelhas e ganham energia.
4. Movimento consome energia.
5. Energia menor ou igual a zero causa morte.
6. Reprodução divide a energia entre agente pai e agente filho.
7. A grama cresce novamente a cada tick.

## 2. Estados e entidades da simulação

Nesta implementação, temos três componentes principais:

| Componente | Descrição |
|---|---|
| Grama | recurso localizado nos patches da grade |
| Ovelhas | agentes herbívoros que comem grama |
| Lobos | agentes predadores que comem ovelhas |

A grama é representada por uma matriz NumPy. Ovelhas e lobos são representados por listas de agentes Python.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import clear_output, HTML
from matplotlib import animation
import random

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 3. Parâmetros do modelo

Os principais parâmetros são:

- `width` e `height`: dimensões da grade;
- `initial_sheep`: quantidade inicial de ovelhas;
- `initial_wolves`: quantidade inicial de lobos;
- `grass_regrowth_rate`: taxa de crescimento da grama;
- `energy_gain_from_grass`: energia que a ovelha recebe ao comer;
- `energy_gain_from_sheep`: energia que o lobo recebe ao caçar;
- `movement_cost`: energia perdida ao se mover;
- `sheep_reproduction_prob` e `wolf_reproduction_prob`: chances de reprodução.

In [ ]:
width = 60
height = 60

initial_sheep = 120
initial_wolves = 40

initial_sheep_energy = 20
initial_wolf_energy = 30

sheep_movement_cost = 1
wolf_movement_cost = 1

energy_gain_from_grass = 4
energy_gain_from_sheep = 20

grass_max = 10
grass_regrowth_rate = 0.5

sheep_reproduction_prob = 0.04
wolf_reproduction_prob = 0.03

max_ticks = 300

## 4. Classe dos agentes

Cada agente possui:

- posição `x` e `y`;
- energia;
- tipo, podendo ser `sheep` ou `wolf`.

O mundo usa fronteiras periódicas: ao sair por um lado, o agente aparece do lado oposto. Esse comportamento é semelhante ao mundo toroidal comum em modelos NetLogo.

In [ ]:
class Agent:
    def __init__(self, x, y, energy, kind):
        self.x = int(x)
        self.y = int(y)
        self.energy = float(energy)
        self.kind = kind

    def move(self, width, height):
        dx = random.choice([-1, 0, 1])
        dy = random.choice([-1, 0, 1])
        self.x = (self.x + dx) % width
        self.y = (self.y + dy) % height

    def position(self):
        return self.x, self.y

## 5. Função de inicialização

A função `setup()` cria a grama, as ovelhas e os lobos.

A grama começa distribuída aleatoriamente entre `0` e `grass_max`, representando um ambiente heterogêneo.

In [ ]:
def setup(rng):
    grass = rng.uniform(0, grass_max, size=(height, width))

    sheep = [
        Agent(
            x=rng.integers(0, width),
            y=rng.integers(0, height),
            energy=initial_sheep_energy,
            kind="sheep"
        )
        for _ in range(initial_sheep)
    ]

    wolves = [
        Agent(
            x=rng.integers(0, width),
            y=rng.integers(0, height),
            energy=initial_wolf_energy,
            kind="wolf"
        )
        for _ in range(initial_wolves)
    ]

    return grass, sheep, wolves

grass, sheep, wolves = setup(rng)
len(sheep), len(wolves), grass.mean()

## 6. Visualização do ambiente

A visualização mostra:

- fundo em escala verde: quantidade de grama;
- pontos claros: ovelhas;
- marcadores `x`: lobos.

In [ ]:
def plot_world(grass, sheep, wolves, tick=0, title="Wolf Sheep Simple 3"):
    plt.figure(figsize=(8, 7))
    plt.imshow(grass, origin="lower", vmin=0, vmax=grass_max)

    if sheep:
        plt.scatter([s.x for s in sheep], [s.y for s in sheep], s=12, label="Ovelhas", alpha=0.8)

    if wolves:
        plt.scatter([w.x for w in wolves], [w.y for w in wolves], s=22, marker="x", label="Lobos", alpha=0.9)

    plt.title(f"{title} — tick {tick}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend(loc="upper right")
    plt.show()

plot_world(grass, sheep, wolves, tick=0, title="Estado inicial")

## 7. Regras das ovelhas

A cada tick, cada ovelha:

1. move-se aleatoriamente;
2. perde energia pelo movimento;
3. come grama se houver grama suficiente no patch;
4. pode se reproduzir;
5. morre se a energia for menor ou igual a zero.

In [ ]:
def update_sheep(sheep, grass):
    survivors = []
    newborns = []

    for s in sheep:
        s.move(width, height)
        s.energy -= sheep_movement_cost

        y, x = s.y, s.x
        if grass[y, x] >= energy_gain_from_grass:
            grass[y, x] -= energy_gain_from_grass
            s.energy += energy_gain_from_grass

        if random.random() < sheep_reproduction_prob and s.energy > 2:
            child_energy = s.energy / 2
            s.energy = s.energy / 2
            newborns.append(Agent(s.x, s.y, child_energy, "sheep"))

        if s.energy > 0:
            survivors.append(s)

    survivors.extend(newborns)
    return survivors

## 8. Regras dos lobos

A cada tick, cada lobo:

1. move-se aleatoriamente;
2. perde energia pelo movimento;
3. procura ovelhas na mesma posição;
4. se encontrar uma ovelha, remove essa ovelha e ganha energia;
5. pode se reproduzir;
6. morre se a energia for menor ou igual a zero.

In [ ]:
def update_wolves(wolves, sheep):
    sheep_by_pos = {}
    for i, s in enumerate(sheep):
        sheep_by_pos.setdefault(s.position(), []).append(i)

    eaten = set()
    survivors = []
    newborns = []

    for w in wolves:
        w.move(width, height)
        w.energy -= wolf_movement_cost

        pos = w.position()
        candidates = sheep_by_pos.get(pos, [])
        candidates = [idx for idx in candidates if idx not in eaten]

        if candidates:
            chosen = candidates[0]
            eaten.add(chosen)
            w.energy += energy_gain_from_sheep

        if random.random() < wolf_reproduction_prob and w.energy > 2:
            child_energy = w.energy / 2
            w.energy = w.energy / 2
            newborns.append(Agent(w.x, w.y, child_energy, "wolf"))

        if w.energy > 0:
            survivors.append(w)

    remaining_sheep = [s for i, s in enumerate(sheep) if i not in eaten]
    survivors.extend(newborns)
    return survivors, remaining_sheep, len(eaten)

## 9. Crescimento da grama

A grama cresce em todos os patches até atingir `grass_max`.

Essa regra representa a recuperação do recurso primário do sistema.

In [ ]:
def regrow_grass(grass):
    grass = grass + grass_regrowth_rate
    grass[grass > grass_max] = grass_max
    return grass

## 10. Execução da simulação

A função `run_simulation()` executa o modelo e registra as principais métricas:

- número de ovelhas;
- número de lobos;
- grama média;
- energia média das ovelhas;
- energia média dos lobos;
- quantidade de ovelhas predadas no tick.

In [ ]:
def run_simulation(max_ticks=300, show_every=50, save_snapshots=True):
    local_rng = np.random.default_rng(RANDOM_SEED)
    random.seed(RANDOM_SEED)

    grass, sheep, wolves = setup(local_rng)
    history = []
    snapshots = []

    for tick in range(max_ticks + 1):
        row = {
            "tick": tick,
            "sheep": len(sheep),
            "wolves": len(wolves),
            "mean_grass": float(np.mean(grass)),
            "mean_sheep_energy": float(np.mean([s.energy for s in sheep])) if sheep else 0,
            "mean_wolf_energy": float(np.mean([w.energy for w in wolves])) if wolves else 0,
            "eaten_sheep": 0,
        }
        history.append(row)

        if save_snapshots and tick % show_every == 0:
            snapshots.append((tick, grass.copy(), [(s.x, s.y) for s in sheep], [(w.x, w.y) for w in wolves]))
            clear_output(wait=True)
            plot_world(grass, sheep, wolves, tick=tick)

        if len(sheep) == 0 and len(wolves) == 0:
            print(f"Todas as populações foram extintas no tick {tick}.")
            break

        sheep = update_sheep(sheep, grass)
        wolves, sheep, eaten = update_wolves(wolves, sheep)
        grass = regrow_grass(grass)

        history[-1]["eaten_sheep"] = eaten

    return pd.DataFrame(history), grass, sheep, wolves, snapshots

## 11. Rodando o modelo

In [ ]:
df_history, final_grass, final_sheep, final_wolves, snapshots = run_simulation(
    max_ticks=max_ticks,
    show_every=50,
    save_snapshots=True
)

df_history.head()

## 12. Resultado final

A visualização final mostra o estado do ambiente ao final da simulação.

In [ ]:
plot_world(final_grass, final_sheep, final_wolves, tick=int(df_history["tick"].iloc[-1]), title="Estado final")

## 13. Evolução temporal das populações

Este gráfico permite observar a dinâmica predador-presa.

Em muitos cenários, surgem ciclos:

- ovelhas aumentam quando há grama suficiente;
- lobos aumentam quando há muitas ovelhas;
- ovelhas diminuem quando a predação cresce;
- lobos diminuem quando faltam ovelhas.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_history["tick"], df_history["sheep"], label="Ovelhas")
plt.plot(df_history["tick"], df_history["wolves"], label="Lobos")
plt.xlabel("Tick")
plt.ylabel("População")
plt.title("Dinâmica populacional")
plt.legend()
plt.grid(True)
plt.show()

## 14. Evolução da grama média

A grama é o recurso de base. Quando ela se esgota, as ovelhas perdem energia. Se as ovelhas diminuem, os lobos também são afetados.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_history["tick"], df_history["mean_grass"], label="Grama média")
plt.xlabel("Tick")
plt.ylabel("Grama média")
plt.title("Recuperação e consumo da grama")
plt.legend()
plt.grid(True)
plt.show()

## 15. Energia média dos agentes

A energia funciona como estoque interno. Ela aumenta quando o agente come e diminui quando se move.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_history["tick"], df_history["mean_sheep_energy"], label="Energia média das ovelhas")
plt.plot(df_history["tick"], df_history["mean_wolf_energy"], label="Energia média dos lobos")
plt.xlabel("Tick")
plt.ylabel("Energia média")
plt.title("Energia média por população")
plt.legend()
plt.grid(True)
plt.show()

## 16. Predação ao longo do tempo

Este gráfico mostra quantas ovelhas foram comidas por lobos em cada tick.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_history["tick"], df_history["eaten_sheep"], label="Ovelhas predadas")
plt.xlabel("Tick")
plt.ylabel("Quantidade")
plt.title("Predação por tick")
plt.legend()
plt.grid(True)
plt.show()

## 17. Snapshots da simulação

Os snapshots mostram o estado do sistema em diferentes ticks.

In [ ]:
for tick, grass_snapshot, sheep_pos, wolf_pos in snapshots:
    fake_sheep = [Agent(x, y, 1, "sheep") for x, y in sheep_pos]
    fake_wolves = [Agent(x, y, 1, "wolf") for x, y in wolf_pos]
    plot_world(grass_snapshot, fake_sheep, fake_wolves, tick=tick, title="Snapshot")

## 18. Métricas finais

In [ ]:
summary = {
    "ticks_executados": int(df_history["tick"].iloc[-1]),
    "ovelhas_finais": int(df_history["sheep"].iloc[-1]),
    "lobos_finais": int(df_history["wolves"].iloc[-1]),
    "grama_media_final": float(df_history["mean_grass"].iloc[-1]),
    "max_ovelhas": int(df_history["sheep"].max()),
    "max_lobos": int(df_history["wolves"].max()),
    "total_ovelhas_predadas": int(df_history["eaten_sheep"].sum()),
}

pd.DataFrame([summary])

## 19. Experimentos comparativos

Agora vamos executar múltiplos cenários alterando alguns parâmetros. Isso ajuda a comparar como o sistema responde a mudanças no ambiente.

In [ ]:
def run_with_parameters(
    scenario_name,
    sheep_reproduction=None,
    wolf_reproduction=None,
    regrowth=None,
    wolf_gain=None,
    ticks=250
):
    global sheep_reproduction_prob, wolf_reproduction_prob, grass_regrowth_rate, energy_gain_from_sheep

    old_values = (
        sheep_reproduction_prob,
        wolf_reproduction_prob,
        grass_regrowth_rate,
        energy_gain_from_sheep,
    )

    if sheep_reproduction is not None:
        sheep_reproduction_prob = sheep_reproduction
    if wolf_reproduction is not None:
        wolf_reproduction_prob = wolf_reproduction
    if regrowth is not None:
        grass_regrowth_rate = regrowth
    if wolf_gain is not None:
        energy_gain_from_sheep = wolf_gain

    df, _, _, _, _ = run_simulation(max_ticks=ticks, show_every=999999, save_snapshots=False)
    df["scenario"] = scenario_name

    sheep_reproduction_prob, wolf_reproduction_prob, grass_regrowth_rate, energy_gain_from_sheep = old_values
    return df

scenarios = pd.concat([
    run_with_parameters("base"),
    run_with_parameters("grama lenta", regrowth=0.2),
    run_with_parameters("ovelhas reproduzem mais", sheep_reproduction=0.08),
    run_with_parameters("lobos mais eficientes", wolf_gain=30),
], ignore_index=True)

scenarios.head()

In [ ]:
plt.figure(figsize=(11, 6))
for name, group in scenarios.groupby("scenario"):
    plt.plot(group["tick"], group["sheep"], label=f"Ovelhas — {name}")
plt.xlabel("Tick")
plt.ylabel("Ovelhas")
plt.title("Comparação entre cenários — população de ovelhas")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
for name, group in scenarios.groupby("scenario"):
    plt.plot(group["tick"], group["wolves"], label=f"Lobos — {name}")
plt.xlabel("Tick")
plt.ylabel("Lobos")
plt.title("Comparação entre cenários — população de lobos")
plt.legend()
plt.grid(True)
plt.show()

## 20. Interpretação

O modelo mostra uma característica central dos sistemas complexos: regras locais simples podem gerar comportamentos globais difíceis de prever.

Alguns padrões esperados:

- Se a grama cresce lentamente, as ovelhas tendem a diminuir.
- Se há poucas ovelhas, os lobos também diminuem.
- Se a reprodução das ovelhas é alta, pode ocorrer explosão populacional.
- Se os lobos ganham muita energia por caça, podem crescer rapidamente e causar colapso das ovelhas.
- Pequenas alterações nos parâmetros podem mudar completamente o comportamento final.

## 21. Próximas extensões possíveis

O modelo pode ser ampliado de várias formas:

1. Adicionar idade dos agentes.
2. Criar velocidade diferente para lobos e ovelhas.
3. Inserir obstáculos no ambiente.
4. Adicionar visão local para os lobos perseguirem ovelhas.
5. Adicionar reprodução dependente de energia mínima.
6. Medir estabilidade, colapso e ciclos populacionais.
7. Comparar o modelo Python com o NetLogo original.
8. Exportar resultados para CSV.
9. Criar animação da simulação.

## 22. Exportando resultados

A tabela histórica pode ser salva para análise posterior em Pandas, Power BI ou outro software.

In [ ]:
df_history.to_csv("wolf_sheep_simple_3_resultados.csv", index=False)
print("Arquivo salvo: wolf_sheep_simple_3_resultados.csv")

## 23. Animação simples

A célula abaixo cria uma animação a partir dos snapshots. Caso o ambiente não suporte animação HTML, use os gráficos estáticos das seções anteriores.

In [ ]:
def animate_snapshots(snapshots):
    fig, ax = plt.subplots(figsize=(7, 7))

    def update(frame):
        ax.clear()
        tick, grass_snapshot, sheep_pos, wolf_pos = frame
        ax.imshow(grass_snapshot, origin="lower", vmin=0, vmax=grass_max)
        if sheep_pos:
            xs, ys = zip(*sheep_pos)
            ax.scatter(xs, ys, s=12, label="Ovelhas", alpha=0.8)
        if wolf_pos:
            xs, ys = zip(*wolf_pos)
            ax.scatter(xs, ys, s=22, marker="x", label="Lobos", alpha=0.9)
        ax.set_title(f"Wolf Sheep Simple 3 — tick {tick}")
        ax.set_xlim(0, width)
        ax.set_ylim(0, height)
        ax.legend(loc="upper right")

    anim = animation.FuncAnimation(fig, update, frames=snapshots, interval=800, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_snapshots(snapshots)